# Test OCR Extraction Pipeline

Walk through the full extraction pipeline step-by-step:
1. Connect to the vLLM server
2. Upload/load a sample document
3. Extract text (digital PDF) or render pages (scanned)
4. Send to LLM for structured parsing
5. Inspect the JSON output

Run this notebook in the `ocr-setup` workspace after deploying `qwen2-5-vl-7b-instruct`.

## 1. Configuration

Update `LLM_BASE_URL` to match your vLLM endpoint.

In [ ]:
import os

# Update this to your vLLM endpoint
LLM_BASE_URL = os.environ.get(
    "LLM_BASE_URL",
    "http://qwen2-5-vl-7b-instruct.runai-jupyter-endemann01.svc.cluster.local/v1"
)
VLM_MODEL = os.environ.get("VLM_MODEL", "Qwen/Qwen2.5-VL-7B-Instruct")

print(f"LLM endpoint: {LLM_BASE_URL}")
print(f"VLM model: {VLM_MODEL}")

## 2. Verify vLLM is running

In [ ]:
import httpx

resp = httpx.get(f"{LLM_BASE_URL}/models", timeout=10.0)
models = resp.json()
print("Available models:")
for m in models.get("data", []):
    print(f"  - {m['id']}")

MODEL_ID = models["data"][0]["id"]
print(f"\nUsing: {MODEL_ID}")

## 3. Load a sample document

Upload a PDF via Jupyter's file browser (drag-and-drop), then set the path below.

In [ ]:
from pathlib import Path

# Set this to your uploaded file
DOC_PATH = Path("sample.pdf")  # <-- UPDATE THIS

assert DOC_PATH.exists(), f"File not found: {DOC_PATH}"
print(f"Document: {DOC_PATH} ({DOC_PATH.stat().st_size / 1024:.0f} KB)")

## 4. Extract text from the PDF

Try PyMuPDF text extraction first. If the page has extractable text,
it's a digital PDF and we can skip the VLM entirely.

In [ ]:
import fitz  # PyMuPDF

doc = fitz.open(str(DOC_PATH))
print(f"Pages: {len(doc)}\n")

for i, page in enumerate(doc):
    text = page.get_text("text").strip()
    has_text = len(text) >= 50
    status = "DIGITAL (text found)" if has_text else "SCANNED (no text)"
    print(f"Page {i+1}: {status} ({len(text)} chars)")
    if has_text:
        print(f"  Preview: {text[:200]}...\n")

doc.close()

## 5a. Digital path: Send extracted text to LLM

If you see "DIGITAL" above, this path applies. The LLM parses the
already-extracted text into structured JSON.

In [ ]:
import time

# Extract text from first page
doc = fitz.open(str(DOC_PATH))
page_text = doc[0].get_text("text").strip()
doc.close()

prompt = """Parse the following document text from a grant award notice.

Return a JSON object with these fields (omit any not present):
  "document_type", "award_number", "sponsor", "pi", "institution",
  "project_title", "award_amount", "project_start", "project_end",
  "fa_rate", "additional_fields"

Preserve ALL dollar amounts and dates exactly. Output only valid JSON."""

full_prompt = f"{prompt}\n\n---\nDOCUMENT TEXT:\n---\n{page_text}"

t0 = time.time()
resp = httpx.post(
    f"{LLM_BASE_URL}/chat/completions",
    json={
        "model": MODEL_ID,
        "messages": [{"role": "user", "content": full_prompt}],
        "max_tokens": 4096,
        "temperature": 0.0,
    },
    timeout=120.0,
)
elapsed = time.time() - t0

result = resp.json()["choices"][0]["message"]["content"]
print(f"Extraction took {elapsed:.1f}s\n")
print(result)

## 5b. Scanned path: Render page as image, send to VLM

If you see "SCANNED" in step 4, this path applies. The VLM reads
the page image directly and does OCR + structuring in one shot.

In [ ]:
import base64
import io
from PIL import Image

# Render first page at 2x resolution
doc = fitz.open(str(DOC_PATH))
page = doc[0]
mat = fitz.Matrix(2.0, 2.0)
pix = page.get_pixmap(matrix=mat)
img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
doc.close()

print(f"Rendered page: {img.width}x{img.height}")

# Display in notebook
img.resize((img.width // 2, img.height // 2))

In [ ]:
# Encode image as base64
buf = io.BytesIO()
img.save(buf, format="PNG")
b64 = base64.b64encode(buf.getvalue()).decode()

vlm_prompt = """Extract all information from this scanned document.

Return a JSON object with these fields (omit any not present):
  "document_type", "award_number", "sponsor", "pi", "institution",
  "project_title", "award_amount", "project_start", "project_end",
  "fa_rate", "additional_fields"

Preserve ALL dollar amounts and dates exactly. Output only valid JSON."""

t0 = time.time()
resp = httpx.post(
    f"{LLM_BASE_URL}/chat/completions",
    json={
        "model": MODEL_ID,
        "messages": [{
            "role": "user",
            "content": [
                {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{b64}"}},
                {"type": "text", "text": vlm_prompt},
            ],
        }],
        "max_tokens": 4096,
        "temperature": 0.0,
    },
    timeout=120.0,
)
elapsed = time.time() - t0

result = resp.json()["choices"][0]["message"]["content"]
print(f"VLM OCR took {elapsed:.1f}s\n")
print(result)

## 6. Parse and inspect the JSON

In [ ]:
import json

try:
    parsed = json.loads(result)
    print(json.dumps(parsed, indent=2))
except json.JSONDecodeError as e:
    print(f"Not valid JSON: {e}")
    print(f"\nRaw output:\n{result}")

## 7. Try different formats

If the output doesn't have the fields you need, try different prompts.
The batch script supports these formats: `award`, `budget`, `terms`,
`key_values`, `table`, `text`, `markdown`, `json`.

Edit the prompt above and re-run, or try the `key_values` format
which is more flexible:

In [ ]:
kv_prompt = """Extract all labeled data points from this document as key-value pairs.
Look for field labels, line items, reference numbers, dates, names, and their values.
Return a JSON object where keys are the field names and values are their values.
Preserve ALL values exactly. Output only valid JSON."""

# Use whichever path is appropriate:
# For digital: swap kv_prompt into the text prompt in step 5a
# For scanned: swap kv_prompt into vlm_prompt in step 5b
print("Copy the prompt above into step 5a or 5b and re-run.")

## Next steps

Once you're happy with the output format:

1. **Batch processing:** Use `batch_extract.py` to process all your docs:
   ```bash
   cd /tmp/KohakuRAG_UI
   python ocr_app/scripts/batch_extract.py \
       --input-dir /home/jovyan/sample_docs \
       --output-dir /home/jovyan/extracted \
       --format award \
       --concurrency 1
   ```

2. **Scale up:** See [batch-processing.md](../docs/runai/batch-processing.md)
   for the production workspace setup.